In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
import matplotlib.pyplot as plt

In [ ]:
!pip install idx2numpy -q
import idx2numpy

In [ ]:
train_image_path="/kaggle/input/datasets/hojjatk/mnist-dataset/train-images.idx3-ubyte"
train_image_labels="/kaggle/input/datasets/hojjatk/mnist-dataset/train-labels.idx1-ubyte"

test_image_path = "/kaggle/input/datasets/hojjatk/mnist-dataset/t10k-images.idx3-ubyte"
test_image_labels = "/kaggle/input/datasets/hojjatk/mnist-dataset/t10k-labels.idx1-ubyte"

In [ ]:
x = idx2numpy.convert_from_file(train_image_path)
y = idx2numpy.convert_from_file(train_image_labels) 
x_test  = idx2numpy.convert_from_file(test_image_path)
y_test  = idx2numpy.convert_from_file(test_image_labels)
x.shape

In [ ]:
x = x.reshape(60000, 784).astype(np.float32) / 255.0
x_test = x_test.reshape(10000, 784).astype(np.float32) / 255.0

x_train = x[5000:].T
y_train = y[5000:]

x_val=x[:5000].T
y_val=y[:5000]


print(f"train: {x_train.shape}")


In [ ]:
def init_parameter():
    w1=np.random.rand(10, 784)-0.5 # generating the weight in range -0.5 to 0.5 in shape 10, 784
    b1=np.random.rand(10,1) - 0.5
    w2=np.random.rand(10, 10)-0.5
    b2=np.random.rand(10,1) - 0.5
    return w1, b1, w2, b2

def ReLU(Z):
    return np.maximum(0,Z)

def softmax(z):
    e = np.exp(z - np.max(z, axis=0, keepdims=True))
    return e / np.sum(e, axis=0, keepdims=True)

def forward_prop(w1, b1, w2, b2, x):
    z1 = w1.dot(x) + b1
    A1 = ReLU(z1)
    z2 = w2.dot(A1) + b2
    A2 = softmax(z2) 
    return z1,A1,z2,A2

def one_hot(Y):
    #creates a tensor of size told
    one_hot_y=np.zeros((Y.size,Y.max()+1)) # size of label , total number of output possible
    #masking than giving them value 1
    one_hot_y[np.arange(Y.size),Y]=1
    #transposing
    one_hot_y=one_hot_y.T
    return one_hot_y

def deriv_ReLU(Z):
    return Z>0
    
def back_prop(z1, A1, z2, A2, w2, X, Y):
    m=Y.size
    one_hot_y= one_hot(Y)
    dz2 = A2 - one_hot_y

    dw2 = (1/m) * dz2.dot(A1.T)
    db2 = (1/m) * np.sum(dz2,axis=1, keepdims=True)
    dz1 =  w2.T.dot(dz2) * deriv_ReLU(z1)
    dw1 = 1 / m * dz1.dot(X.T)
    db1 = 1 / m * np.sum(dz1,axis=1, keepdims=True)
    return dw1 , db1, dw2, db2

def update_params(w1,b1,w2,b2, dw1,db1,dw2,db2,alpha):
    w1 = w1 - alpha * dw1
    b1 = b1 - alpha * db1
    w2 = w2 - alpha * dw2
    b2 = b2 - alpha * db2
    return w1, b1, w2, b2

In [ ]:
def get_predictions(a2):
    return np.argmax(a2, 0)# gives array conatining max element column wise

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions ==Y )/ Y.size

def gradient_descent(X,Y, iterations, alpha):
    w1,b1,w2,b2 = init_parameter()
    for i in range(iterations):
        z1, a1, z2, a2 = forward_prop(w1,b1,w2,b2,X)
        dw1,db1, dw2, db2 = back_prop(z1, a1, z2, a2, w2, X, Y )
        w1,b1,w2,b2= update_params(w1, b1, w2, b2, dw1, db1, dw2, db2, alpha)
        if i% 50 == 0:
            print("iteration:", i)
            print("accuracy: ", get_accuracy(get_predictions(a2),Y) )
    return w1, b1, w2, b2

In [ ]:
w1, b1, w2, b2= gradient_descent(x_train,y_train, 500,0.1)

In [ ]:
def make_predictions(X, w1, b1, w2, b2):
    _, _, _, a2 = forward_prop(w1, b1, w2, b2, X)
    predictions = get_predictions(a2)
    return predictions

def test_prediction(index, w1, b1, w2, b2):
    current_image = x_train[:, index, None]
    prediction = make_predictions(x_train[:, index, None], w1, b1, w2, b2)
    label = y_train[index]
    print("Prediction:", prediction)
    print("Label:", label)

    current_image = current_image.reshape((28, 28)) * 255
    plt.gray()
    plt.imshow(current_image, interpolation='nearest')
    plt.show()

In [ ]:
test_prediction(0, w1, b1, w2, b2)
test_prediction(1, w1, b1, w2, b2)
test_prediction(2, w1, b1, w2, b2)

In [ ]:
dev_predictions = make_predictions(x_val, w1, b1, w2, b2)
get_accuracy(dev_predictions, y_val)